In [2]:
import pandas as pd
print("Pandas is working!")

Pandas is working!


In [5]:
accounts = pd.read_csv(r'C:\Users\praso\Downloads\archive\ravenstack_accounts.csv')
subscriptions = pd.read_csv(r'C:\Users\praso\Downloads\archive\ravenstack_subscriptions.csv')
feature_usage = pd.read_csv(r'C:\Users\praso\Downloads\archive\ravenstack_feature_usage.csv')
support_tickets = pd.read_csv(r'C:\Users\praso\Downloads\archive\ravenstack_support_tickets.csv')
churn_events = pd.read_csv(r'C:\Users\praso\Downloads\archive\ravenstack_churn_events.csv')
print("All files loaded successfully")
print("Accounts shape:", accounts.shape)
print("Subscriptions shape:", subscriptions.shape)
print("Feature usage shape:", feature_usage.shape)
print("Support tickets shape:", support_tickets.shape)
print("Churn events shape:", churn_events.shape)

All files loaded successfully
Accounts shape: (500, 10)
Subscriptions shape: (5000, 14)
Feature usage shape: (25000, 8)
Support tickets shape: (2000, 9)
Churn events shape: (600, 9)


In [6]:
# Convert date columns to proper datetime format
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'], format='%d-%m-%Y')

subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])  # stays NaT if blank

feature_usage['usage_date'] = pd.to_datetime(feature_usage['usage_date'])

support_tickets['submitted_at'] = pd.to_datetime(support_tickets['submitted_at'])
support_tickets['closed_at'] = pd.to_datetime(support_tickets['closed_at'])

churn_events['churn_date'] = pd.to_datetime(churn_events['churn_date'])

print("Dates converted successfully")
accounts.dtypes

Dates converted successfully


account_id                 object
account_name               object
industry                   object
country                    object
signup_date        datetime64[ns]
referral_source            object
plan_tier                  object
seats                       int64
is_trial                     bool
churn_flag                   bool
dtype: object

In [7]:
print("Accounts signup_date range:", accounts['signup_date'].min(), "to", accounts['signup_date'].max())
print("Subscriptions start_date range:", subscriptions['start_date'].min(), "to", subscriptions['start_date'].max())
print("\nNull end_date count (still-active subscriptions):", subscriptions['end_date'].isnull().sum())

Accounts signup_date range: 2023-01-02 00:00:00 to 2024-12-31 00:00:00
Subscriptions start_date range: 2023-01-09 00:00:00 to 2024-12-31 00:00:00

Null end_date count (still-active subscriptions): 4514


In [8]:
print("Duplicate account_ids:", accounts['account_id'].duplicated().sum())
print("Duplicate subscription_ids:", subscriptions['subscription_id'].duplicated().sum())

Duplicate account_ids: 0
Duplicate subscription_ids: 0


In [9]:
import datetime as dt

# Reference date = "today" for the dataset (use the max date seen, since this is historical data)
snapshot_date = subscriptions['start_date'].max()

# is_active: True if end_date is null (still running) as of snapshot
subscriptions['is_active'] = subscriptions['end_date'].isnull()

# tenure_days: how long the subscription has run (or ran, if churned)
subscriptions['tenure_days'] = (
    subscriptions['end_date'].fillna(snapshot_date) - subscriptions['start_date']
).dt.days

print("Snapshot date used:", snapshot_date)
subscriptions[['subscription_id', 'start_date', 'end_date', 'is_active', 'tenure_days']].head()

Snapshot date used: 2024-12-31 00:00:00


,subscription_id,start_date,end_date,is_active,tenure_days
0,S-8cec59,2023-12-23,2024-04-12,False,111
1,S-0f6f44,2024-06-11,NaT,True,203
2,S-51c0d1,2024-11-25,NaT,True,36
3,S-f81687,2024-11-23,2024-12-13,False,20
4,S-cff5a2,2024-01-10,NaT,True,356


In [13]:
accounts['cohort_month'] = accounts['signup_date'].dt.to_period('M').astype(str)

accounts[['account_id', 'signup_date', 'cohort_month']].head()

,account_id,signup_date,cohort_month
0,A-2e4581,2024-10-16,2024-10
1,A-43a9e3,2023-08-17,2023-08
2,A-0a282f,2024-08-27,2024-08
3,A-1f0ac7,2023-08-27,2023-08
4,A-ce550d,2024-10-27,2024-10


In [14]:
print("Tenure days - min:", subscriptions['tenure_days'].min(), 
      "max:", subscriptions['tenure_days'].max(), 
      "mean:", round(subscriptions['tenure_days'].mean(), 1))

print("\nActive subscriptions:", subscriptions['is_active'].sum())
print("Churned subscriptions:", (~subscriptions['is_active']).sum())

Tenure days - min: 0 max: 722 mean: 160.9

Active subscriptions: 4514
Churned subscriptions: 486


In [15]:
support_rollup = support_tickets.groupby('account_id').agg(
    total_tickets=('ticket_id', 'count'),
    avg_satisfaction_score=('satisfaction_score', 'mean'),
    avg_resolution_time_hours=('resolution_time_hours', 'mean'),
    escalation_count=('escalation_flag', 'sum'),
    urgent_ticket_count=('priority', lambda x: (x == 'urgent').sum())
).reset_index()

support_rollup['escalation_rate'] = support_rollup['escalation_count'] / support_rollup['total_tickets']

print("Accounts with support tickets:", support_rollup.shape[0])
support_rollup.head()

Accounts with support tickets: 492


,account_id,total_tickets,avg_satisfaction_score,avg_resolution_time_hours,escalation_count,urgent_ticket_count,escalation_rate
0,A-00bed1,4,4.0,31.750000,0,2,0.0
1,A-00cac8,2,NaN,33.000000,0,2,0.0
2,A-0158bb,1,3.0,32.000000,0,1,0.0
3,A-016043,3,4.0,30.333333,0,0,0.0
4,A-019782,2,3.0,10.000000,0,2,0.0


In [16]:
# Map subscription_id -> account_id
sub_to_account = subscriptions[['subscription_id', 'account_id']]

usage_with_account = feature_usage.merge(sub_to_account, on='subscription_id', how='left')

usage_rollup = usage_with_account.groupby('account_id').agg(
    total_usage_events=('usage_id', 'count'),
    total_usage_count=('usage_count', 'sum'),
    avg_usage_duration_secs=('usage_duration_secs', 'mean'),
    total_errors=('error_count', 'sum'),
    distinct_features_used=('feature_name', 'nunique')
).reset_index()

usage_rollup['error_rate'] = usage_rollup['total_errors'] / usage_rollup['total_usage_count']

print("Accounts with usage data:", usage_rollup.shape[0])
usage_rollup.head()

Accounts with usage data: 500


,account_id,total_usage_events,total_usage_count,avg_usage_duration_secs,total_errors,distinct_features_used,error_rate
0,A-00bed1,51,514,2818.313725,27,32,0.052529
1,A-00cac8,58,602,2954.586207,31,30,0.051495
2,A-0158bb,36,364,3390.305556,22,19,0.060440
3,A-016043,47,490,2810.106383,21,26,0.042857
4,A-019782,55,562,2924.509091,30,28,0.053381


In [19]:
# Get latest/current subscription per account (most recent start_date)
latest_sub = subscriptions.sort_values('start_date').groupby('account_id').tail(1)

latest_sub_cols = latest_sub[['account_id', 'subscription_id', 'plan_tier', 'mrr_amount', 
                                'arr_amount', 'is_active', 'tenure_days', 'upgrade_flag', 
                                'downgrade_flag', 'billing_frequency', 'auto_renew_flag']]

# Merge account info + latest subscription + support rollup + usage rollup
master = accounts.merge(latest_sub_cols, on='account_id', how='left')
master = master.merge(support_rollup, on='account_id', how='left')
master = master.merge(usage_rollup, on='account_id', how='left')

# Fill missing support/usage stats with sensible defaults (accounts with no tickets = 0 tickets, neutral satisfaction)
master['total_tickets'] = master['total_tickets'].fillna(0)
master['escalation_rate'] = master['escalation_rate'].fillna(0)
master['avg_satisfaction_score'] = master['avg_satisfaction_score'].fillna(master['avg_satisfaction_score'].mean())
master['error_rate'] = master['error_rate'].fillna(master['error_rate'].mean())
master['total_usage_count'] = master['total_usage_count'].fillna(0)
master['distinct_features_used'] = master['distinct_features_used'].fillna(0)

print("Master table shape:", master.shape)
master.head()

Master table shape: (500, 33)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier_x,seats,is_trial,churn_flag,...,avg_resolution_time_hours,escalation_count,urgent_ticket_count,escalation_rate,total_usage_events,total_usage_count,avg_usage_duration_secs,total_errors,distinct_features_used,error_rate
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,23.000000,0.0,1.0,0.000000,55,535,2769.800000,38,27,0.071028
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,...,38.000000,0.0,2.0,0.000000,35,355,2889.600000,14,23,0.039437
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,43.666667,0.0,1.0,0.000000,83,821,3026.626506,48,34,0.058465
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False,...,29.000000,0.0,0.0,0.000000,41,382,2500.682927,21,26,0.054974
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True,...,42.285714,1.0,2.0,0.142857,58,579,3720.327586,31,32,0.053541


In [20]:
print("Nulls remaining per column:")
print(master.isnull().sum()[master.isnull().sum() > 0])

Nulls remaining per column:
avg_resolution_time_hours    8
escalation_count             8
urgent_ticket_count          8
dtype: int64


In [21]:
# Fill remaining support-related nulls (accounts with 0 tickets)
master['avg_resolution_time_hours'] = master['avg_resolution_time_hours'].fillna(0)
master['escalation_count'] = master['escalation_count'].fillna(0)
master['urgent_ticket_count'] = master['urgent_ticket_count'].fillna(0)

# Resolve duplicate plan_tier: keep the subscription-level one (current plan), drop the account-level one
master = master.drop(columns=['plan_tier_x'])
master = master.rename(columns={'plan_tier_y': 'plan_tier'})

print("Nulls remaining:", master.isnull().sum().sum())
print("Columns:", master.shape[1])
master[['account_id', 'plan_tier', 'mrr_amount', 'is_active']].head()

Nulls remaining: 0
Columns: 32


,account_id,plan_tier,mrr_amount,is_active
0,A-2e4581,Basic,836,True
1,A-43a9e3,Pro,882,True
2,A-0a282f,Pro,98,True
3,A-1f0ac7,Pro,1176,True
4,A-ce550d,Enterprise,21691,False


In [22]:
# Normalize each signal to 0-1 scale so they can be combined fairly
def normalize(series, invert=False):
    norm = (series - series.min()) / (series.max() - series.min())
    return 1 - norm if invert else norm

master['norm_usage'] = normalize(master['total_usage_count'])
master['norm_features'] = normalize(master['distinct_features_used'])
master['norm_satisfaction'] = normalize(master['avg_satisfaction_score'])
master['norm_error_rate'] = normalize(master['error_rate'], invert=True)       # lower error = healthier
master['norm_escalation'] = normalize(master['escalation_rate'], invert=True)  # lower escalation = healthier
master['norm_tenure'] = normalize(master['tenure_days'])

# Weighted Customer Health Score (0-100 scale)
# Usage & feature adoption matter most (engagement), then support experience, then tenure
master['health_score'] = (
    master['norm_usage'] * 25 +
    master['norm_features'] * 20 +
    master['norm_satisfaction'] * 20 +
    master['norm_error_rate'] * 15 +
    master['norm_escalation'] * 10 +
    master['norm_tenure'] * 10
).round(1)

print("Health score range:", master['health_score'].min(), "-", master['health_score'].max())
print("Mean health score:", master['health_score'].mean().round(1))

master[['account_id', 'plan_tier', 'is_active', 'health_score']].sort_values('health_score').head()

Health score range: 16.8 - 81.6
Mean health score: 50.2


,account_id,plan_tier,is_active,health_score
181,A-5b051a,Pro,True,16.8
311,A-3d957b,Enterprise,True,19.2
177,A-56962b,Enterprise,True,22.1
462,A-f19b24,Basic,True,22.2
354,A-b54c01,Basic,True,22.4


In [23]:
# Revenue at Risk = MRR exposure for ACTIVE accounts with low health scores
# Define risk tiers based on health score
def risk_tier(score):
    if score < 33:
        return 'High Risk'
    elif score < 60:
        return 'Medium Risk'
    else:
        return 'Low Risk'

master['risk_tier'] = master['health_score'].apply(risk_tier)

# Revenue at risk = MRR of active accounts only, weighted by inverse health score
# (lower health = more of their MRR counted as "at risk")
master['risk_weight'] = (100 - master['health_score']) / 100
master['revenue_at_risk'] = master.apply(
    lambda row: row['mrr_amount'] * row['risk_weight'] if row['is_active'] else 0, axis=1
).round(2)

print("Total MRR (active accounts):", master.loc[master['is_active'], 'mrr_amount'].sum())
print("Total Revenue at Risk:", master['revenue_at_risk'].sum().round(2))
print()
print("Risk tier breakdown (active accounts only):")
print(master[master['is_active']]['risk_tier'].value_counts())
print()
print("Revenue at risk by tier:")
print(master[master['is_active']].groupby('risk_tier')['mrr_amount'].sum())

Total MRR (active accounts): 1078303
Total Revenue at Risk: 536305.01

Risk tier breakdown (active accounts only):
risk_tier
Medium Risk    359
Low Risk        73
High Risk       22
Name: count, dtype: int64

Revenue at risk by tier:
risk_tier
High Risk       63917
Low Risk       198828
Medium Risk    815558
Name: mrr_amount, dtype: int64


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

features = ['tenure_days', 'health_score', 'total_tickets', 'escalation_rate', 
            'error_rate', 'total_usage_count', 'distinct_features_used', 
            'avg_satisfaction_score', 'seats']

X = master[features]
y = master['churn_flag'].astype(int)

print("Churn class balance:")
print(y.value_counts(normalize=True).round(3))

# Scale features so no single column dominates
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# class_weight='balanced' forces the model to pay attention to the minority (churned) class
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

master['churn_probability'] = model.predict_proba(X_scaled)[:, 1]
master['renewal_probability'] = (1 - master['churn_probability']).round(3)

y_pred_proba = model.predict_proba(X_test)[:, 1]
print("\nModel AUC score:", round(roc_auc_score(y_test, y_pred_proba), 3))
print()
print(classification_report(y_test, model.predict(X_test), zero_division=0))

master[['account_id', 'health_score', 'churn_flag', 'renewal_probability']].sort_values('renewal_probability').head()

Churn class balance:
churn_flag
0    0.78
1    0.22
Name: proportion, dtype: float64

Model AUC score: 0.581

              precision    recall  f1-score   support

           0       0.81      0.44      0.57        78
           1       0.24      0.64      0.35        22

    accuracy                           0.48       100
   macro avg       0.53      0.54      0.46       100
weighted avg       0.68      0.48      0.52       100



,account_id,health_score,churn_flag,renewal_probability
34,A-bad8c1,59.1,False,0.240
260,A-151c9a,50.8,False,0.303
85,A-157070,50.6,False,0.318
478,A-6dee43,59.6,False,0.328
146,A-4c6f11,53.4,True,0.334


## Interpreting the Churn Model Results

The logistic regression trained on engagement and support features (tenure, health score, total tickets, escalation rate, error rate, usage count, distinct features used, satisfaction score, seats) achieved an **AUC of 0.581** — only marginally better than a random classifier (AUC 0.5).

Rather than treating this as a failed model to discard, I investigated *why* the signal was weak, since that diagnosis is itself useful information:

**1. Feature correlation with churn was uniformly low.** No single engineered feature correlated with `churn_flag` above |r| = 0.09 — meaning none of the behavioral signals I built (usage volume, support interactions, satisfaction) meaningfully separated churned from retained accounts on their own.

**2. Churn reasons are fragmented, not concentrated.** A breakdown of `reason_code` in the churn events table shows six causes within a tight 4-point band: features (19.0%), support (17.3%), budget (17.3%), unknown (15.8%), competitor (15.3%), pricing (15.2%). There is no dominant driver.

**3. A follow-up structural test (SQL Query 6) confirmed the same pattern.** Testing whether upgrades/downgrades in the 90 days preceding churn were a stronger signal than day-to-day behavior showed the same fragmentation — no single structural event reliably precedes churn either.

### Why this matters

If churn were driven by one dominant cause (e.g., "everyone churns because of low usage"), a single retention lever would fix most of it, and a behavioral model would likely have found strong signal. The fact that it didn't — combined with the even spread of reason codes — indicates churn here is **driven by account-specific, situational factors** rather than a systemic product or engagement problem.

**This finding directly shaped the dashboard design**: instead of a single blanket retention campaign, the Save Campaign Prioritizer assigns **segmented, rules-based recommended actions** per account (e.g., "Escalate to CS Manager" for high-escalation accounts, "Feature Adoption Nudge" for low feature adoption), since a one-size-fits-all approach would be a poor fit for this kind of fragmented churn pattern.


In [29]:
correlations = master[features + ['churn_flag']].corr()['churn_flag'].sort_values(key=abs, ascending=False)
print("Correlation with churn_flag:")
print(correlations)

print("\n--- Model coefficients (which features the model actually weighted) ---")
coef_df = pd.DataFrame({
    'feature': features,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print(coef_df)

Correlation with churn_flag:
churn_flag                1.000000
health_score              0.094292
error_rate               -0.087450
distinct_features_used    0.067676
total_usage_count         0.064186
escalation_rate           0.063825
tenure_days               0.062109
avg_satisfaction_score    0.034402
seats                    -0.033437
total_tickets            -0.020425
Name: churn_flag, dtype: float64

--- Model coefficients (which features the model actually weighted) ---
                  feature  coefficient
6  distinct_features_used     0.301021
4              error_rate    -0.241454
5       total_usage_count    -0.171392
0             tenure_days     0.103067
8                   seats    -0.087803
1            health_score     0.068534
7  avg_satisfaction_score    -0.055828
3         escalation_rate     0.051608
2           total_tickets    -0.006494


In [31]:
print("Churn reason breakdown:")
print(churn_events['reason_code'].value_counts())
print("\nChurn reason % of total churns:")
print(churn_events['reason_code'].value_counts(normalize=True).round(3))

Churn reason breakdown:
reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64

Churn reason % of total churns:
reason_code
features      0.190
support       0.173
budget        0.173
unknown       0.158
competitor    0.153
pricing       0.152
Name: proportion, dtype: float64


In [33]:
import os

output_folder = r'C:\Users\praso\Downloads\archive\clean_exports'
os.makedirs(output_folder, exist_ok=True)

master.to_csv(f'{output_folder}\\dim_fact_master_account.csv', index=False)
subscriptions.to_csv(f'{output_folder}\\fact_subscription.csv', index=False)
feature_usage.to_csv(f'{output_folder}\\fact_usage.csv', index=False)
support_tickets.to_csv(f'{output_folder}\\fact_support.csv', index=False)
churn_events.to_csv(f'{output_folder}\\fact_churn.csv', index=False)

print("All clean tables exported to:", output_folder)
print("Files created:")
for f in os.listdir(output_folder):
    print(" -", f)

All clean tables exported to: C:\Users\praso\Downloads\archive\clean_exports
Files created:
 - dim_fact_master_account.csv
 - fact_churn.csv
 - fact_subscription.csv
 - fact_support.csv
 - fact_usage.csv
